In [1]:
import os
# Only go up a directory if we are currently in the 'research' folder
if os.getcwd().endswith('research'):
    os.chdir("../")

In [2]:
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

In [3]:
%pwd

'/Users/manish/Desktop/chicken-disease-classification'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareCallbacksConfig:
  root_dir: Path
  tensorboard_root_log_dir: Path
  checkpoint_model_filepath: Path

In [12]:
from chicken_disease_classification.constants import *
from chicken_disease_classification.utils.common import read_yaml, create_directories

class ConfigurationManager:
  def __init__(
    self,
    config_filepath = CONFIG_FILE_PATH,
    params_filepath = PARAMS_FILE_PATH):

    self.config = read_yaml(config_filepath)
    self.params = read_yaml(params_filepath)

    create_directories([self.config.artifacts_root])

  
  def get_prepare_callback_config(self) -> PrepareCallbacksConfig:
    config = self.config.prepare_callbacks
    model_ckpt_dir = os.path.dirname(config.checkpoint_model_filepath)
    create_directories([
      Path(model_ckpt_dir),
      Path(config.tensorboard_root_log_dir)
    ])

    prepare_callback_config = PrepareCallbacksConfig(
      root_dir=Path(config.root_dir),
      tensorboard_root_log_dir=Path(config.tensorboard_root_log_dir),
      checkpoint_model_filepath=Path(config.checkpoint_model_filepath)
    )

    return prepare_callback_config

In [23]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

class PrepareCallback:
  def __init__(self, config: PrepareCallbacksConfig):
    self.config = config

  @property
  def _create_tb_callbacks(self):
    timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
    tb_running_log_dir = os.path.join(
      str(self.config.tensorboard_root_log_dir),
      f"tb_logs_at_{timestamp}"
    )
    return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)

  @property
  def _create_ckpt_callbacks(self):
    return tf.keras.callbacks.ModelCheckpoint(
      filepath=str(self.config.checkpoint_model_filepath),
      save_best_only=True
    )

  def get_tb_ckpt_callbacks(self):
    return [
      self._create_tb_callbacks,
      self._create_ckpt_callbacks
    ]

In [24]:
try:
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callback_config()
    prepare_callbacks = PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()
    print("Callbacks created successfully:", callback_list)
except Exception as e:
    raise e

[2026-09-29 19:34:25,662: INFO: chicken_disease_classification: yaml file: config/config.yaml loaded successfully]
[2026-09-29 19:34:25,665: INFO: chicken_disease_classification: yaml file: params.yaml loaded successfully]
[2026-09-29 19:34:25,665: INFO: chicken_disease_classification: Created directory at: artifacts]
[2026-09-29 19:34:25,666: INFO: chicken_disease_classification: Created directory at: artifacts/prepare_callbacks/checkpoint_dir]
[2026-09-29 19:34:25,667: INFO: chicken_disease_classification: Created directory at: artifacts/prepare_callbacks/tensorboard_root_log_dir]
Callbacks created successfully: [<keras.src.callbacks.tensorboard.TensorBoard object at 0x14ee7ef50>, <keras.src.callbacks.model_checkpoint.ModelCheckpoint object at 0x14e901e50>]


In [25]:
import time
timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
f"tb_logs_at_{timestamp}"


'tb_logs_at_2026-09-29-19-35-41'